# sesac_power_station_2 DB loader

This notebook loads the 12 CSV files in `dalykit/data/erd_v2_csv` into MySQL using the revised ERD.

Run from top to bottom.

What this notebook does:

1. Connect to MySQL.
2. Create the 12 tables from the revised ERD.
3. Check that the 12 CSV files exist and that their columns match the table columns.
4. Empty the target tables before loading, so rerunning the notebook does not duplicate rows.
5. Load all 12 CSV files in foreign-key order.
6. Check row counts, foreign keys, and sensor duplicate keys.
7. Show each real table with `SELECT *`, without joins.

In [ ]:
from pathlib import Path
import time

import numpy as np
import pandas as pd
import pymysql
from IPython.display import display

# ---------------------------------------------------------------------
# 1) Find the project folder automatically.
#
# This notebook is inside dalykit/data/data_code, but Jupyter can run it
# from different working directories. This loop walks upward until it
# finds the folder that contains dalykit/data.
# ---------------------------------------------------------------------
cwd = Path.cwd().resolve()
PROJECT_ROOT = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / 'dalykit' / 'data').exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise RuntimeError('Project root not found. Run this notebook inside the final_project workspace.')

# ---------------------------------------------------------------------
# 2) CSV folder.
#
# These are the 12 CSV files generated for the revised ERD.
# Example: tb_power_station.csv, tb_mac_a_sensor.csv, tb_prediction.csv
# ---------------------------------------------------------------------
CSV_DIR = PROJECT_ROOT / 'dalykit' / 'data' / 'erd_v2_csv'

# ---------------------------------------------------------------------
# 3) MySQL connection settings.
#
# Change only user/password/database if your local MySQL uses different
# values. PyMySQL works with MySQL even though the package name is pymysql.
# ---------------------------------------------------------------------
DB_CONFIG = {
    'host': 'localhost',
    'port': 3306,
    'user': 'root',
    'password': 'test1234',
    'database': 'sesac_power_station_db',
    'charset': 'utf8mb4',
    'local_infile': True,
}

# ---------------------------------------------------------------------
# 4) Safety switches.
#
# RECREATE_TABLES = True
#   Drops the 12 target tables and creates them again.
#   This is useful when the table schema changed.
#
# USE_LOAD_DATA = True
#   Uses MySQL's fast CSV loader: LOAD DATA LOCAL INFILE.
#   If MySQL blocks local_infile, set this to False and rerun.
#   False is slower, but uses normal Python chunk inserts.
# ---------------------------------------------------------------------
RECREATE_TABLES = True
USE_LOAD_DATA = True

print('PROJECT_ROOT =', PROJECT_ROOT)
print('CSV_DIR =', CSV_DIR)
print('CSV exists =', CSV_DIR.exists())

## 1. Connect to MySQL

This cell creates the database if it does not exist, then checks that MySQL is reachable.

In [ ]:
def get_conn(database=True, autocommit=False):
    """Create a MySQL connection.

    database=True means connect directly to sesac_power_station_db.
    database=False is used once at the beginning, because the database
    may not exist yet.
    """
    config = DB_CONFIG.copy()
    if not database:
        config.pop('database', None)
    return pymysql.connect(**config, autocommit=autocommit)

# Create the database itself first. This does not create any tables yet.
conn = get_conn(database=False, autocommit=True)
try:
    with conn.cursor() as cur:
        cur.execute(
            f"CREATE DATABASE IF NOT EXISTS `{DB_CONFIG['database']}` "
            "CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci"
        )
finally:
    conn.close()

# Simple connection test. If this prints database name and MySQL version,
# the connection settings are correct.
conn = get_conn()
try:
    with conn.cursor() as cur:
        cur.execute('SELECT DATABASE(), VERSION()')
        print(cur.fetchone())
finally:
    conn.close()

## 1-1. Check MySQL local infile setting

Fast CSV loading uses `LOAD DATA LOCAL INFILE`.

If `local_infile` is `OFF`, MySQL will refuse the fast loader. In that case either:

- Enable `local_infile` in MySQL, or
- Set `USE_LOAD_DATA = False` in the first code cell and rerun from the table creation step.

In [ ]:
# MySQL server-side setting check for LOAD DATA LOCAL INFILE.
conn = get_conn()
try:
    with conn.cursor() as cur:
        cur.execute("SHOW VARIABLES LIKE 'local_infile'")
        local_infile_status = cur.fetchone()
        print('local_infile =', local_infile_status)
finally:
    conn.close()

if local_infile_status and str(local_infile_status[1]).upper() != 'ON':
    print('local_infile is OFF. Either enable it in MySQL, or set USE_LOAD_DATA = False in the first cell.')

## 2. Create revised ERD tables

This section defines the 12 MySQL tables.

Important:

- The table names and column names match the revised ERD CSV files.
- Foreign keys are included.
- Sensor tables include `UNIQUE(part_id, measured_at)`, so the same sensor part cannot have duplicated timestamps.
- SQL `COMMENT` clauses are omitted here because they are not needed for loading or checking the data.

In [ ]:
# DDL statements for dropping and creating the revised ERD tables.
# You normally do not need to edit this cell.

DROP_TABLES_SQL = [
    'DROP TABLE IF EXISTS `tb_alarm_info`',
    'DROP TABLE IF EXISTS `tb_prediction`',
    'DROP TABLE IF EXISTS `tb_model_info`',
    'DROP TABLE IF EXISTS `tb_mac_a_sensor`',
    'DROP TABLE IF EXISTS `tb_mac_b_sensor`',
    'DROP TABLE IF EXISTS `tb_bac_sensor`',
    'DROP TABLE IF EXISTS `tb_dgan_sensor`',
    'DROP TABLE IF EXISTS `tb_vhp_sensor`',
    'DROP TABLE IF EXISTS `tb_part`',
    'DROP TABLE IF EXISTS `tb_power_plant`',
    'DROP TABLE IF EXISTS `tb_user_info`',
    'DROP TABLE IF EXISTS `tb_power_station`',
]

CREATE_TABLES_SQL = [
    """
    CREATE TABLE IF NOT EXISTS `tb_power_station` (
      `station_id` INT NOT NULL AUTO_INCREMENT,
      `station_name` VARCHAR(255) NOT NULL,
      `station_lat` DOUBLE NULL,
      `station_lon` DOUBLE NULL,
      `station_add` VARCHAR(255) NOT NULL,
      `station_capacity` DOUBLE NOT NULL,
      CONSTRAINT `PK_TB_POWER_STATION` PRIMARY KEY (`station_id`)
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci
    """,
    """
    CREATE TABLE IF NOT EXISTS `tb_power_plant` (
      `plant_id` INT NOT NULL AUTO_INCREMENT,
      `station_id` INT NOT NULL,
      `plant_name` VARCHAR(255) NOT NULL,
      `fuel_type` VARCHAR(255) NOT NULL,
      `plant_status` VARCHAR(255) NOT NULL,
      CONSTRAINT `PK_TB_POWER_PLANT` PRIMARY KEY (`plant_id`),
      CONSTRAINT `FK_POWER_PLANT_STATION`
        FOREIGN KEY (`station_id`) REFERENCES `tb_power_station` (`station_id`)
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci
    """,
    """
    CREATE TABLE IF NOT EXISTS `tb_part` (
      `part_id` INT NOT NULL AUTO_INCREMENT,
      `plant_id` INT NOT NULL,
      `part_code` INT NOT NULL,
      `part_type` VARCHAR(255) NOT NULL,
      `part_name` VARCHAR(255) NOT NULL,
      CONSTRAINT `PK_TB_PART` PRIMARY KEY (`part_id`),
      CONSTRAINT `FK_PART_PLANT`
        FOREIGN KEY (`plant_id`) REFERENCES `tb_power_plant` (`plant_id`)
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci
    """,
    """
    CREATE TABLE IF NOT EXISTS `tb_user_info` (
      `user_num` INT NOT NULL AUTO_INCREMENT,
      `email` VARCHAR(255) NOT NULL,
      `password_hash` VARCHAR(255) NOT NULL,
      `name` VARCHAR(255) NOT NULL,
      `permission` VARCHAR(255) NOT NULL,
      `team_name` VARCHAR(255) NULL,
      `created_at` DATETIME NOT NULL,
      `assigned_station` VARCHAR(255) NULL,
      CONSTRAINT `PK_TB_USER_INFO` PRIMARY KEY (`user_num`)
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci
    """,
    """
    CREATE TABLE IF NOT EXISTS `tb_model_info` (
      `model_id` INT NOT NULL AUTO_INCREMENT,
      `part_id` INT NOT NULL,
      `model_name` VARCHAR(255) NOT NULL,
      `model_version` VARCHAR(255) NULL,
      `threshold` DOUBLE NOT NULL,
      CONSTRAINT `PK_TB_MODEL_INFO` PRIMARY KEY (`model_id`),
      CONSTRAINT `FK_MODEL_INFO_PART`
        FOREIGN KEY (`part_id`) REFERENCES `tb_part` (`part_id`)
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci
    """,
    """
    CREATE TABLE IF NOT EXISTS `tb_prediction` (
      `prediction_id` INT NOT NULL AUTO_INCREMENT,
      `model_id` INT NOT NULL,
      `anomaly_score` DOUBLE NOT NULL,
      `prediction_time` DATETIME NOT NULL,
      CONSTRAINT `PK_TB_PREDICTION` PRIMARY KEY (`prediction_id`),
      CONSTRAINT `FK_PREDICTION_MODEL`
        FOREIGN KEY (`model_id`) REFERENCES `tb_model_info` (`model_id`)
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci
    """,
    """
    CREATE TABLE IF NOT EXISTS `tb_alarm_info` (
      `alarm_id` INT NOT NULL AUTO_INCREMENT,
      `user_num` INT NOT NULL,
      `prediction_id` INT NOT NULL,
      `alarm_content` TEXT NOT NULL,
      `alarm_type` VARCHAR(10) NOT NULL,
      `alarm_create_at` DATETIME NOT NULL,
      CONSTRAINT `PK_TB_ALARM_INFO` PRIMARY KEY (`alarm_id`),
      CONSTRAINT `FK_ALARM_USER`
        FOREIGN KEY (`user_num`) REFERENCES `tb_user_info` (`user_num`),
      CONSTRAINT `FK_ALARM_PREDICTION`
        FOREIGN KEY (`prediction_id`) REFERENCES `tb_prediction` (`prediction_id`)
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci
    """,
]

SENSOR_DDL = {
    'tb_mac_a_sensor': ('mac_a_id', 'PK_TB_MAC_A_SENSOR', 'FK_MAC_A_SENSOR_PART', 'UK_MAC_A_SENSOR_PART_TIME'),
    'tb_mac_b_sensor': ('mac_b_id', 'PK_TB_MAC_B_SENSOR', 'FK_MAC_B_SENSOR_PART', 'UK_MAC_B_SENSOR_PART_TIME'),
    'tb_bac_sensor': ('bac_id', 'PK_TB_BAC_SENSOR', 'FK_BAC_SENSOR_PART', 'UK_BAC_SENSOR_PART_TIME'),
    'tb_dgan_sensor': ('dgan_id', 'PK_TB_DGAN_SENSOR', 'FK_DGAN_SENSOR_PART', 'UK_DGAN_SENSOR_PART_TIME'),
    'tb_vhp_sensor': ('vhp_id', 'PK_TB_VHP_SENSOR', 'FK_VHP_SENSOR_PART', 'UK_VHP_SENSOR_PART_TIME'),
}

for table, (pk_col, pk_name, fk_name, uk_name) in SENSOR_DDL.items():
    CREATE_TABLES_SQL.append(f"""
    CREATE TABLE IF NOT EXISTS `{table}` (
      `{pk_col}` BIGINT NOT NULL AUTO_INCREMENT,
      `part_id` INT NOT NULL,
      `measured_at` DATETIME NOT NULL,
      `current` DOUBLE NULL,
      `NDE_temp` DOUBLE NULL,
      `DE_temp` DOUBLE NULL,
      `NDE_X_vibration` DOUBLE NULL,
      `NDE_Y_vibration` DOUBLE NULL,
      `DE_X_vibration` DOUBLE NULL,
      `DE_Y_vibration` DOUBLE NULL,
      CONSTRAINT `{pk_name}` PRIMARY KEY (`{pk_col}`),
      CONSTRAINT `{fk_name}`
        FOREIGN KEY (`part_id`) REFERENCES `tb_part` (`part_id`),
      CONSTRAINT `{uk_name}` UNIQUE (`part_id`, `measured_at`)
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci
    """)

In [ ]:
def recreate_schema():
    """Create the 12 target tables.

    If RECREATE_TABLES is True, the existing 12 tables are dropped first.
    This means rerunning the whole notebook starts from a clean schema.
    """
    conn = get_conn()
    try:
        with conn.cursor() as cur:
            # Temporarily disable FK checks so child tables can be dropped first.
            cur.execute('SET FOREIGN_KEY_CHECKS = 0')
            if RECREATE_TABLES:
                for sql in DROP_TABLES_SQL:
                    cur.execute(sql)
            for sql in CREATE_TABLES_SQL:
                cur.execute(sql)
            cur.execute('SET FOREIGN_KEY_CHECKS = 1')
        conn.commit()
    finally:
        conn.close()

recreate_schema()
print('Schema is ready')

## 3. Check CSV files and row counts

Before loading into MySQL, this cell checks two things:

1. Every expected CSV file exists.
2. The CSV column order exactly matches the table column order.

This catches most mistakes before the database insert step.

In [ ]:
# Expected column order for each table.
# These lists must match both the MySQL table definitions and the CSV headers.
TABLE_COLUMNS = {
    'tb_power_station': ['station_id', 'station_name', 'station_lat', 'station_lon', 'station_add', 'station_capacity'],
    'tb_power_plant': ['plant_id', 'station_id', 'plant_name', 'fuel_type', 'plant_status'],
    'tb_part': ['part_id', 'plant_id', 'part_code', 'part_type', 'part_name'],
    'tb_user_info': ['user_num', 'email', 'password_hash', 'name', 'permission', 'team_name', 'created_at', 'assigned_station'],
    'tb_model_info': ['model_id', 'part_id', 'model_name', 'model_version', 'threshold'],
    'tb_mac_a_sensor': ['mac_a_id', 'part_id', 'measured_at', 'current', 'NDE_temp', 'DE_temp', 'NDE_X_vibration', 'NDE_Y_vibration', 'DE_X_vibration', 'DE_Y_vibration'],
    'tb_mac_b_sensor': ['mac_b_id', 'part_id', 'measured_at', 'current', 'NDE_temp', 'DE_temp', 'NDE_X_vibration', 'NDE_Y_vibration', 'DE_X_vibration', 'DE_Y_vibration'],
    'tb_bac_sensor': ['bac_id', 'part_id', 'measured_at', 'current', 'NDE_temp', 'DE_temp', 'NDE_X_vibration', 'NDE_Y_vibration', 'DE_X_vibration', 'DE_Y_vibration'],
    'tb_dgan_sensor': ['dgan_id', 'part_id', 'measured_at', 'current', 'NDE_temp', 'DE_temp', 'NDE_X_vibration', 'NDE_Y_vibration', 'DE_X_vibration', 'DE_Y_vibration'],
    'tb_vhp_sensor': ['vhp_id', 'part_id', 'measured_at', 'current', 'NDE_temp', 'DE_temp', 'NDE_X_vibration', 'NDE_Y_vibration', 'DE_X_vibration', 'DE_Y_vibration'],
    'tb_prediction': ['prediction_id', 'model_id', 'anomaly_score', 'prediction_time'],
    'tb_alarm_info': ['alarm_id', 'user_num', 'prediction_id', 'alarm_content', 'alarm_type', 'alarm_create_at'],
}

# Load order matters because of foreign keys.
# Parent tables must be inserted before child tables.
LOAD_ORDER = [
    'tb_power_station', 'tb_power_plant', 'tb_part', 'tb_user_info', 'tb_model_info',
    'tb_mac_a_sensor', 'tb_mac_b_sensor', 'tb_bac_sensor', 'tb_dgan_sensor', 'tb_vhp_sensor',
    'tb_prediction', 'tb_alarm_info',
]

CSV_FILES = {table: CSV_DIR / f'{table}.csv' for table in LOAD_ORDER}

def count_csv_rows(path: Path) -> int:
    """Count data rows in a CSV file, excluding the header row."""
    with path.open('rb') as f:
        return sum(1 for _ in f) - 1

summary = []
for table in LOAD_ORDER:
    path = CSV_FILES[table]
    if not path.exists():
        raise FileNotFoundError(path)

    # Read only the header. This is fast even for large CSV files.
    csv_cols = list(pd.read_csv(path, nrows=0).columns)
    expected_cols = TABLE_COLUMNS[table]
    if csv_cols != expected_cols:
        raise ValueError(f'{table} column mismatch\nCSV: {csv_cols}\nEXPECTED: {expected_cols}')

    summary.append({'table': table, 'csv_path': str(path), 'csv_rows': count_csv_rows(path)})

csv_summary_df = pd.DataFrame(summary)
display(csv_summary_df)

## 4. Load CSV files

This section loads the CSV files into MySQL.

The notebook uses two safety measures against duplicate loading:

1. If `RECREATE_TABLES=True`, the 12 tables are dropped and recreated earlier.
2. Right before loading, `truncate_tables()` empties all 12 tables again.

So running the whole notebook again should not double the data.

In [ ]:
def truncate_tables():
    """Empty all 12 target tables before loading.

    This prevents duplicate rows if the notebook is run more than once.
    Tables are truncated in reverse FK order so child tables are emptied first.
    """
    conn = get_conn()
    try:
        with conn.cursor() as cur:
            cur.execute('SET FOREIGN_KEY_CHECKS = 0')
            for table in reversed(LOAD_ORDER):
                cur.execute(f'TRUNCATE TABLE `{table}`')
            cur.execute('SET FOREIGN_KEY_CHECKS = 1')
        conn.commit()
    finally:
        conn.close()


def detect_line_ending(path: Path) -> str:
    """Detect whether the CSV uses Windows CRLF or Unix LF line endings."""
    sample = path.read_bytes()[:8192]
    return '\\r\\n' if b'\r\n' in sample else '\\n'


def load_csv_with_load_data(table: str, path: Path):
    """Fast loader using MySQL LOAD DATA LOCAL INFILE."""
    columns = TABLE_COLUMNS[table]
    col_sql = ', '.join(f'`{col}`' for col in columns)
    line_ending = detect_line_ending(path)
    sql = f"""
        LOAD DATA LOCAL INFILE %s
        INTO TABLE `{table}`
        CHARACTER SET utf8mb4
        FIELDS TERMINATED BY ',' OPTIONALLY ENCLOSED BY '"'
        LINES TERMINATED BY '{line_ending}'
        IGNORE 1 LINES
        ({col_sql})
    """
    conn = get_conn()
    try:
        with conn.cursor() as cur:
            cur.execute(sql, (path.as_posix(),))
        conn.commit()
    finally:
        conn.close()


def load_csv_with_chunks(table: str, path: Path, chunksize=50_000):
    """Fallback loader using pandas chunks and INSERT statements.

    This is slower than LOAD DATA LOCAL INFILE, but it works when MySQL
    does not allow local infile loading.
    """
    columns = TABLE_COLUMNS[table]
    placeholders = ', '.join(['%s'] * len(columns))
    col_sql = ', '.join(f'`{col}`' for col in columns)
    sql = f'INSERT INTO `{table}` ({col_sql}) VALUES ({placeholders})'

    total = 0
    conn = get_conn()
    try:
        with conn.cursor() as cur:
            for chunk in pd.read_csv(path, chunksize=chunksize, low_memory=False):
                # Convert pandas NaN to Python None so MySQL receives NULL.
                chunk = chunk.replace({np.nan: None})
                rows = [tuple(row) for row in chunk.itertuples(index=False, name=None)]
                cur.executemany(sql, rows)
                total += len(rows)
                print(f'  {table}: {total:,} rows')
        conn.commit()
    finally:
        conn.close()


def load_one_table(table: str):
    """Load one CSV file into its matching MySQL table."""
    path = CSV_FILES[table]
    start = time.time()
    if USE_LOAD_DATA:
        load_csv_with_load_data(table, path)
    else:
        load_csv_with_chunks(table, path)
    elapsed = time.time() - start
    print(f'{table} loaded: {count_csv_rows(path):,} rows, {elapsed:.1f} sec')

In [ ]:
# Empty the 12 tables first, then load every CSV in FK-safe order.
truncate_tables()
print('Existing data truncated')

for table in LOAD_ORDER:
    load_one_table(table)

print('All tables loaded')

## 5. Validate row counts

This checks whether each MySQL table has the same number of rows as its CSV file.

In [ ]:
def fetch_df(sql, params=None):
    """Run a SELECT query and return the result as a pandas DataFrame."""
    conn = get_conn()
    try:
        return pd.read_sql(sql, conn, params=params)
    finally:
        conn.close()

count_rows = []
conn = get_conn()
try:
    with conn.cursor() as cur:
        for table in LOAD_ORDER:
            cur.execute(f'SELECT COUNT(*) FROM `{table}`')
            db_rows = cur.fetchone()[0]
            csv_rows = count_csv_rows(CSV_FILES[table])
            count_rows.append({'table': table, 'csv_rows': csv_rows, 'db_rows': db_rows, 'match': csv_rows == db_rows})
finally:
    conn.close()

count_check_df = pd.DataFrame(count_rows)
display(count_check_df)

if not count_check_df['match'].all():
    raise AssertionError('Some DB row counts do not match CSV row counts.')

## 6. Validate foreign keys and sensor unique keys

This section checks data integrity after loading.

- FK checks: child rows must point to existing parent rows.
- Sensor unique checks: each sensor table must not duplicate `(part_id, measured_at)`.

In [ ]:
# Each query returns the number of broken links.
# Every bad_rows value should be 0.

fk_checks = {
    'tb_power_plant.station_id -> tb_power_station.station_id': """
        SELECT COUNT(*) FROM tb_power_plant p
        LEFT JOIN tb_power_station s ON p.station_id = s.station_id
        WHERE s.station_id IS NULL
    """,
    'tb_part.plant_id -> tb_power_plant.plant_id': """
        SELECT COUNT(*) FROM tb_part p
        LEFT JOIN tb_power_plant pp ON p.plant_id = pp.plant_id
        WHERE pp.plant_id IS NULL
    """,
    'tb_model_info.part_id -> tb_part.part_id': """
        SELECT COUNT(*) FROM tb_model_info m
        LEFT JOIN tb_part p ON m.part_id = p.part_id
        WHERE p.part_id IS NULL
    """,
    'tb_prediction.model_id -> tb_model_info.model_id': """
        SELECT COUNT(*) FROM tb_prediction pr
        LEFT JOIN tb_model_info m ON pr.model_id = m.model_id
        WHERE m.model_id IS NULL
    """,
    'tb_alarm_info.user_num -> tb_user_info.user_num': """
        SELECT COUNT(*) FROM tb_alarm_info a
        LEFT JOIN tb_user_info u ON a.user_num = u.user_num
        WHERE u.user_num IS NULL
    """,
    'tb_alarm_info.prediction_id -> tb_prediction.prediction_id': """
        SELECT COUNT(*) FROM tb_alarm_info a
        LEFT JOIN tb_prediction p ON a.prediction_id = p.prediction_id
        WHERE p.prediction_id IS NULL
    """,
}

for table in ['tb_mac_a_sensor', 'tb_mac_b_sensor', 'tb_bac_sensor', 'tb_dgan_sensor', 'tb_vhp_sensor']:
    fk_checks[f'{table}.part_id -> tb_part.part_id'] = f"""
        SELECT COUNT(*) FROM {table} s
        LEFT JOIN tb_part p ON s.part_id = p.part_id
        WHERE p.part_id IS NULL
    """

check_rows = []
conn = get_conn()
try:
    with conn.cursor() as cur:
        for name, sql in fk_checks.items():
            cur.execute(sql)
            bad_rows = cur.fetchone()[0]
            check_rows.append({'check': name, 'bad_rows': bad_rows, 'ok': bad_rows == 0})
finally:
    conn.close()

fk_check_df = pd.DataFrame(check_rows)
display(fk_check_df)

if not fk_check_df['ok'].all():
    raise AssertionError('FK validation failed.')

In [ ]:
# Check duplicated sensor timestamps for each of the five sensor tables.
# duplicate_rows should be 0 for every table.

unique_rows = []
conn = get_conn()
try:
    with conn.cursor() as cur:
        for table in ['tb_mac_a_sensor', 'tb_mac_b_sensor', 'tb_bac_sensor', 'tb_dgan_sensor', 'tb_vhp_sensor']:
            cur.execute(f'''
                SELECT
                    COUNT(*) AS total_rows,
                    COUNT(DISTINCT part_id, measured_at) AS unique_part_time,
                    MIN(measured_at) AS min_time,
                    MAX(measured_at) AS max_time
                FROM `{table}`
            ''')
            total_rows, unique_part_time, min_time, max_time = cur.fetchone()
            unique_rows.append({
                'table': table,
                'total_rows': total_rows,
                'unique_part_time': unique_part_time,
                'duplicate_rows': total_rows - unique_part_time,
                'min_time': min_time,
                'max_time': max_time,
            })
finally:
    conn.close()

sensor_unique_df = pd.DataFrame(unique_rows)
display(sensor_unique_df)

if (sensor_unique_df['duplicate_rows'] != 0).any():
    raise AssertionError('A sensor table has duplicated (part_id, measured_at) rows.')

## 7. Show each loaded table as-is

This section shows the actual contents of each table using only `SELECT *`.

No joins are used here.

Large tables have millions of rows, so this displays:

- total row count
- first 5 rows

In [ ]:
# Show each of the 12 real tables.
# This is intentionally SELECT * only. No JOIN, no extra columns.
for table in LOAD_ORDER:
    print('\n' + '=' * 100)
    print(f'TABLE: {table}')

    # Row count proves how much data was loaded.
    count_df = fetch_df(f'SELECT COUNT(*) AS row_count FROM `{table}`')
    display(count_df)

    # LIMIT 5 keeps the notebook responsive for large tables.
    display(fetch_df(f'SELECT * FROM `{table}` ORDER BY 1 LIMIT 5'))

## 8. Optional: show table columns

This checks the actual MySQL column order for the 12 real tables.

No joins are used here either.

In [ ]:
# Show MySQL column metadata for each real table.
for table in LOAD_ORDER:
    print('\n' + '=' * 100)
    print(f'COLUMNS: {table}')
    display(fetch_df(f'SHOW COLUMNS FROM `{table}`'))

In [ ]:
##제발